
# Selected Samples Extraction Workbench v5 — JSON-first Notebook-Only Version

This notebook is for **testing and improving extraction on the selected sample folder before changing backend/core code**.

It does not call the API. It does not write to the database. It does not need Codex.

It gives you:

1. File discovery from `samples/selected samples/`
2. Quality check and rejection status for bad images/files
3. Image preprocessing experiments
4. OCR variants with rotation and PSM modes
5. Best OCR selection
6. Exact extracted text saved per file
7. Post-processing on OCR text
8. Common field extraction
9. Lab / urine / culture / radiology extraction
10. One CSV for document-level summary
11. One CSV where **each extracted part/test/field is saved in its own row**
12. Debug output for weak/failed files

Recommended folder:

```text
your_repo/
  notebooks/selected_samples_extraction_workbench_v3.ipynb
  samples/selected samples/
    file1.pdf
    file2.jpg
    ...
```


In [1]:

# =========================
# 0) Environment and paths
# =========================

from pathlib import Path
import os, re, json, math, shutil, hashlib, tempfile, traceback, unicodedata
from dataclasses import dataclass, asdict, field
from typing import Any, Optional
from collections import defaultdict, Counter

import pandas as pd
import numpy as np

try:
    from PIL import Image, ImageOps, ImageEnhance, ImageFilter
except Exception as e:
    raise RuntimeError("Pillow is required. Install with: pip install pillow") from e

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    TESSERACT_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

# Try to find repo root automatically.
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / "app").exists() or (p / "samples").exists() or (p / ".git").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
SELECTED_DIRS = [
    REPO_ROOT / "samples" / "selected samples",
    #REPO_ROOT / "samples" / "selected_samples",
    #REPO_ROOT / "samples" / "raw",
    #REPO_ROOT / "samples",
    Path("/mnt/data"),
]

OUTPUT_DIR = REPO_ROOT / "notebook_outputs" / "selected_samples_v5"
TEXT_DIR = OUTPUT_DIR / "texts"
DEBUG_DIR = OUTPUT_DIR / "debug"
IMAGE_DEBUG_DIR = DEBUG_DIR / "images"
VARIANT_TEXT_DIR = DEBUG_DIR / "ocr_variants"

for d in [OUTPUT_DIR, TEXT_DIR, DEBUG_DIR, IMAGE_DEBUG_DIR, VARIANT_TEXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".pdf", ".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Tesseract available:", TESSERACT_AVAILABLE)
print("OpenCV available:", CV2_AVAILABLE)
print("PyMuPDF available:", fitz is not None)


Repo root: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents
Output dir: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selected_samples_v5
Tesseract available: True
OpenCV available: True
PyMuPDF available: True



## 1) Discover selected sample files

This cell only selects real files. It never selects a folder.

You can also manually set `ONLY_FILES` if you want to test a small subset first.


In [2]:

# =========================
# 1) Discover files
# =========================

ONLY_FILES = []  # Optional: put absolute/local file paths here to test only a few files.
# Example:
# ONLY_FILES = ["samples/selected samples/0014161672_14041209_O_00404121721.pdf"]

def discover_files() -> list[Path]:
    if ONLY_FILES:
        files = [Path(x).expanduser() for x in ONLY_FILES]
        return [p.resolve() for p in files if p.exists() and p.is_file()]
    found = []
    for folder in SELECTED_DIRS:
        if folder.exists() and folder.is_dir():
            for ext in SUPPORTED_EXTS:
                found.extend(folder.rglob(f"*{ext}"))
                found.extend(folder.rglob(f"*{ext.upper()}"))
    # Deduplicate while preserving order
    seen = set()
    clean = []
    for p in found:
        rp = p.resolve()
        if rp not in seen and rp.is_file():
            seen.add(rp)
            clean.append(rp)
    return clean

FILES = discover_files()
print(f"Found {len(FILES)} selected files")
for i, p in enumerate(FILES[:80]):
    print(f"{i:03d} | {p.name} | {p.suffix} | {p.stat().st_size/1024:.1f} KB")

assert FILES, "No files found. Put PDFs/images in samples/selected samples/ or set ONLY_FILES."


Found 20 selected files
000 | 0014161672_14041209_O_00404121721.pdf | .pdf | 44.2 KB
001 | 0021858845_14041209_O_00404121731.pdf | .pdf | 42.9 KB
002 | 0023471026_14041209_O_00404121728.pdf | .pdf | 40.8 KB
003 | 0025631314_14041209_O_00404121726.pdf | .pdf | 41.2 KB
004 | 0024150010_14041209_O_00404121722.pdf | .pdf | 44.1 KB
005 | 0025692283_14041209_O_00404121730.pdf | .pdf | 42.4 KB
006 | 0020139871_14041209_O_00404121714.pdf | .pdf | 41.4 KB
007 | 20260427_181919.jpg | .jpg | 58.2 KB
008 | ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg | .jpg | 56.8 KB
009 | ۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg | .jpg | 148.6 KB
010 | 20260427_181713.jpg | .jpg | 44.2 KB
011 | -2147483648_-210195.jpg | .jpg | 42.0 KB
012 | -2147483648_-210193.jpg | .jpg | 77.8 KB
013 | ۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg | .jpg | 123.6 KB
014 | 20260427_181636.jpg | .jpg | 38.7 KB
015 | 20260427_181554.jpg | .jpg | 42.5 KB
016 | ۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg | .jpg | 67.8 KB
017 | ۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg | .jpg | 41.8 KB
018 | 20260427_181654.jpg | .jpg | 36.8 KB
019 | ۲۰۲۶۰۴۲۸_۱۵۰۱۲۰.j


## 2) Configuration

Statuses used by this notebook:

| Status | Meaning |
|---|---|
| `extracted_good` | enough text and structured data were extracted |
| `partial_extraction` | OCR worked but structured extraction is weak |
| `needs_manual_review` | text/data exists but confidence is low or fields conflict |
| `poor_quality_rejected` | image/file quality is too poor for reliable OCR |
| `ocr_failed` | OCR produced no usable text |
| `unrelated_or_not_medical` | extracted text does not look like a medical document |
| `invalid_file` | unsupported/corrupt/missing file |


In [3]:

# =========================
# 2) Config
# =========================

CONFIG = {
    "max_pdf_pages": 5,
    "min_ocr_text_len": 20,
    "min_good_ocr_score": 7,
    "poor_quality_min_score": 0.22,
    "good_quality_min_score": 0.60,
    "save_variant_texts": True,
    "save_debug_images": True,
    "run_rotation_variants": True,
    "psm_modes": [6, 4, 11],
    # Tesseract language: try eng+fas on your local machine if Persian pack exists.
    # If it fails, the notebook falls back to eng.
    "tesseract_langs": ["eng+fas", "eng"],
}

MEDICAL_KEYWORDS = [
    "laboratory", "lab", "hematology", "biochemistry", "serology", "hormone", "thyroid",
    "urine", "culture", "bacteriology", "cbc", "wbc", "rbc", "hemoglobin", "platelet",
    "fbs", "glucose", "creatinine", "cholesterol", "triglycerides", "tsh", "ferritin",
    "vitamin", "crp", "pt", "inr", "ptt",
    "آزمایشگاه", "آزمايشگاه", "پاتوبیولوژی", "پاتوبيولوژی", "درمانگاه", "بیمارستان",
    "پزشک", "پذیرش", "نتیجه", "ادرار", "خون",
]

LAB_ALIASES = {
    # CBC
    "WBC": ["WBC", "W.B.C", "White Blood Cell"],
    "RBC": ["RBC", "R.B.C", "Red Blood Cell"],
    "HGB": ["HGB", "Hb", "Hemoglobin", "Haemoglobin", "Hemoglobine"],
    "HCT": ["HCT", "H.C.T", "Hematocrit"],
    "MCV": ["MCV", "M.C.V"],
    "MCH": ["MCH", "M.C.H"],
    "MCHC": ["MCHC", "M.C.H.C"],
    "RDW-CV": ["RDW-CV", "RDW CV", "RDW"],
    "RDW-SD": ["RDW-SD", "RDW SD"],
    "PLT": ["PLT", "Platelet", "Platelets"],
    "PDW": ["PDW"],
    "MPV": ["MPV"],
    "P-LCR": ["P-LCR", "PLCR"],
    "PCT": ["PCT"],
    "NEUT#": ["NEUT#", "Neut#", "Neutrophil#", "Neutrophils#"],
    "LYMPH#": ["LYMPH#", "Lymph#", "Lymphocyte#"],
    "MONO#": ["MONO#", "Mono#", "Monocyte#"],
    "EO#": ["EO#", "EOS#", "Eosinophil#"],
    "BASO#": ["BASO#", "Baso#", "Basophil#"],
    "NEUT%": ["NEUT%", "Neut%", "Neutrophils", "Neutrophil"],
    "LYMPH%": ["LYMPH%", "Lymph%", "Lymphocytes", "Lymphocyte", "Lympho"],
    "MONO%": ["MONO%", "Mono%", "Monocytes", "Monocyte"],
    "EO%": ["EO%", "EOS%", "Eosinophils", "Eosinophil"],
    "BASO%": ["BASO%", "Basophils", "Basophil"],
    "ESR": ["ESR", "Erythrocyte Sediment Rate", "Erythrocyte Sedimentation Rate"],
    # Biochemistry
    "FBS": ["Fasting Blood Glucose", "Fasting blood sugar", "FBS", "Glucose", "Fasting Serum Glucose", "Blood sugar"],
    "HbA1c": ["HbA1c", "Hemoglobin A1c", "Hemoglobine A1c"],
    "EAG": ["Estimated Average Glucose", "EAG"],
    "BUN": ["BUN", "Blood Urea Nitrogen", "Serum Urea Nitrogen"],
    "Urea": ["Urea"],
    "Creatinine": ["Creatinine"],
    "Uric Acid": ["Uric Acid"],
    "Total Cholesterol": ["Total Cholesterol", "Cholesterol", "Cholestrol", "Cholestrol-Total"],
    "Triglycerides": ["Triglycerides", "TG"],
    "HDL": ["HDL", "HDL Cholesterol", "HDL-Cholesterol"],
    "LDL": ["LDL", "LDL Cholesterol", "LDL-Cholesterol", "LDL - Cholesterol", "LDL-Cholesterol (Direct)"],
    "AST": ["AST", "SGOT", "SGOT(AST)", "AST(SGOT)"],
    "ALT": ["ALT", "SGPT", "SGPT(ALT)"],
    "ALP": ["ALP", "Alkaline Phosphatase"],
    "Calcium": ["Calcium", "Calcium (Ca)"],
    "Phosphorus": ["Phosphorus", "P"],
    "Sodium": ["Sodium", "Na"],
    "Potassium": ["Potassium", "K"],
    "Iron": ["Iron", "Fe"],
    "TIBC": ["TIBC", "Total Iron Binding Capacity"],
    "Bilirubin Total": ["Bilirubin Total", "Bilirubin(Total)"],
    "Bilirubin Direct": ["Bilirubin Direct", "Bilirubin(Direct)"],
    # Hormone / special
    "TSH": ["TSH", "Thyroid Stimulating Hormone"],
    "T3": ["T3", "Tri iodothyronine", "Triiodothyronine"],
    "T4": ["T4", "Thyroxine"],
    "Free T4": ["Free T4", "FT4"],
    "Vitamin D": ["Vitamin D", "25 Hydroxy Vitamin D", "25-Hydroxy Vitamin D3", "25-OH Vitamin D3"],
    "Vitamin B12": ["Vitamin B12"],
    "Ferritin": ["Ferritin"],
    "CRP": ["CRP", "C-Reactive protein", "C - Reactive protein"],
    "LH": ["LH"],
    "FSH": ["FSH"],
    "Prolactin": ["Prolactin"],
    "Testosterone": ["Testosterone"],
    "Free Testosterone": ["Free Testosterone"],
    "Estradiol": ["Estradiol"],
    "DHEA-SO4": ["DHEA-SO4", "DHEA SO4"],
    "17OH-Progesterone": ["17OH-Progesterone", "17OH Progesterone", "17OH-Progesteron"],
    # Coagulation
    "PT": ["PT", "Prothrombin Time"],
    "PT Control": ["PT Control"],
    "INR": ["INR"],
    "PTT": ["PTT", "Activated PTT"],
    # Urine / microbiology
    "Urine Culture": ["Urine Culture", "Culture"],
    "Color": ["Color"],
    "Appearance": ["Appearance"],
    "Specific Gravity": ["Specific Gravity", "Sp. Gravity", "Specific Gravit"],
    "pH": ["pH", "PH"],
    "Protein": ["Protein"],
    "Urine Glucose": ["Urine Glucose", "Glucose"],
    "Ketone": ["Ketone"],
    "Bilirubin Urine": ["Bilirubin"],
    "Urobilinogen": ["Urobilinogen"],
    "Nitrite": ["Nitrite"],
    "Blood/Hb": ["Blood/Hb", "Hemoglobin", "Blood"],
    "WBC/HPF": ["WBC/HPF", "W.B.C / HPF", "W.B.C HPF"],
    "RBC/HPF": ["RBC/HPF", "R.B.C / HPF", "R.B.C HPF"],
    "Epithelial Cells": ["Epithelial Cells", "Epithelial Cell"],
    "Bacteria": ["Bacteria"],
    "Mucus": ["Mucus"],
    "Casts": ["Casts", "Cast"],
    "Crystals": ["Crystals", "Crystal"],
}

ALL_ALIAS_PHRASES = [(std, alias) for std, aliases in LAB_ALIASES.items() for alias in aliases]



## 3) Normalization, quality, preview helpers


In [4]:

# =========================
# 3) Helpers
# =========================

DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def normalize_digits(text: str) -> str:
    return (text or "").translate(DIGIT_MAP)

def normalize_persian(text: str) -> str:
    """Normalize Persian/Arabic OCR text, including Arabic presentation forms.

    The selected TAV PDFs often contain Arabic presentation glyphs like ﺳﻦ / دﻛﺘﺮ.
    NFKC normalization converts them to normal Unicode letters so regex extraction works.
    """
    text = unicodedata.normalize("NFKC", text or "")
    text = normalize_digits(text)
    text = text.replace("ي", "ی").replace("ى", "ی").replace("ك", "ک")
    text = text.replace("\u200c", " ").replace("\u200f", " ").replace("\u200e", " ")
    text = re.sub(r"[ـ]+", "", text)
    text = re.sub(r"[^\S\n]+", " ", text)
    return text

def safe_id(path: Path) -> str:
    stem = path.stem
    stem = re.sub(r"[^\w\u0600-\u06FF.-]+", "_", stem)
    if len(stem) > 80:
        stem = stem[:80]
    return stem or hashlib.md5(str(path).encode()).hexdigest()[:10]

def hash_value(value: str | None) -> str | None:
    if not value:
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()

def validate_jalali_date(date: str | None) -> bool:
    if not date:
        return False
    m = re.match(r"^(13[9][0-9]|14[0-1][0-9]|1420)[/-](\d{1,2})[/-](\d{1,2})$", normalize_digits(date))
    if not m:
        return False
    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return 1390 <= y <= 1420 and 1 <= mo <= 12 and 1 <= d <= 31

def file_mime_guess(path: Path) -> str:
    ext = path.suffix.lower()
    return {
        ".pdf": "application/pdf",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
        ".tif": "image/tiff",
        ".tiff": "image/tiff",
    }.get(ext, "application/octet-stream")

def assess_image_quality(path: Path) -> dict:
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        w, h = img.size
        gray = np.array(img.convert("L"))
        mean = float(np.mean(gray))
        std = float(np.std(gray))
        if CV2_AVAILABLE:
            lap = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        else:
            lap = 0.0
        blur_score = min(1.0, lap / 500.0) if lap else 0.5
        brightness_score = max(0.0, 1 - abs(mean - 127) / 127)
        contrast_score = min(1.0, std / 80.0)
        resolution_score = min(1.0, (w * h) / (1000 * 1000))
        overall = max(0.0, min(1.0, blur_score*.35 + brightness_score*.20 + contrast_score*.25 + resolution_score*.20))
        issues = []
        if lap and blur_score < .18:
            issues.append("severe_blur")
        if contrast_score < .25:
            issues.append("low_contrast")
        if mean < 45:
            issues.append("too_dark")
        if mean > 220:
            issues.append("too_bright")
        if resolution_score < .10:
            issues.append("low_resolution")
        if h > w * 1.35 or w > h * 1.35:
            issues.append("possible_rotation_or_sideways")
        if overall >= CONFIG["good_quality_min_score"]:
            status = "good_quality"
        elif overall >= CONFIG["poor_quality_min_score"]:
            status = "needs_preprocessing"
        else:
            status = "poor_quality"
        return {
            "status": status,
            "overall_quality_score": round(overall, 3),
            "issues": issues,
            "width": w,
            "height": h,
            "brightness_mean": round(mean, 2),
            "contrast_std": round(std, 2),
            "laplacian_variance": round(lap, 2),
        }
    except Exception as e:
        return {
            "status": "unreadable",
            "overall_quality_score": 0.0,
            "issues": ["unreadable_image", str(e)],
            "width": None,
            "height": None,
        }

def assess_file_quality(path: Path) -> dict:
    if path.suffix.lower() == ".pdf":
        return {"status": "pdf_quality_not_assessed_here", "overall_quality_score": None, "issues": [], "width": None, "height": None}
    return assess_image_quality(path)



## 4) Image preprocessing variants and OCR line reconstruction

This is notebook-only. It tests better OCR without changing repository code.


In [5]:

# =========================
# 4) OCR variants
# =========================

@dataclass
class OCRCandidate:
    text: str
    confidence: float
    variant: str
    psm: int | None
    lang: str
    score: float
    image_path: str | None = None
    error: str | None = None
    details: dict = field(default_factory=dict)

def pil_to_cv_gray(img: Image.Image):
    arr = np.array(img.convert("L"))
    return arr

def save_debug_image(img: Image.Image, out_path: Path):
    if CONFIG["save_debug_images"]:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(out_path)

def image_variants(path: Path, out_base: Path) -> list[tuple[str, Image.Image]]:
    img0 = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    variants = [("original_oriented", img0)]
    # Light grayscale/autocontrast
    gray = ImageOps.grayscale(img0)
    variants.append(("light_autocontrast", ImageOps.autocontrast(gray).convert("RGB")))
    # Mild contrast
    mild = ImageEnhance.Contrast(gray).enhance(1.4)
    variants.append(("mild_contrast", ImageOps.autocontrast(mild).convert("RGB")))
    # Sharpen light
    variants.append(("light_sharpen", ImageOps.autocontrast(gray.filter(ImageFilter.SHARPEN)).convert("RGB")))
    # Threshold only as fallback
    if CV2_AVAILABLE:
        arr = np.array(gray)
        thr = cv2.adaptiveThreshold(arr,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11)
        variants.append(("threshold_fallback", Image.fromarray(thr).convert("RGB")))
    # Rotations
    if CONFIG["run_rotation_variants"]:
        base = list(variants[:3])
        for name, im in base:
            for deg in [90, 180, 270]:
                variants.append((f"{name}_rot{deg}", im.rotate(deg, expand=True)))
    # Save variant images
    saved = []
    for name, im in variants:
        fp = out_base / f"{name}.png"
        save_debug_image(im, fp)
        saved.append((name, im))
    return saved

def reconstruct_lines_from_tesseract_data(data: dict) -> tuple[str, float, dict]:
    rows = []
    n = len(data.get("text", []))
    for i in range(n):
        txt = str(data["text"][i] or "").strip()
        if not txt:
            continue
        try:
            conf = float(data.get("conf", ["-1"])[i])
        except Exception:
            conf = -1
        if conf < 0:
            continue
        key = (
            data.get("block_num", [0]*n)[i],
            data.get("par_num", [0]*n)[i],
            data.get("line_num", [0]*n)[i],
        )
        rows.append({
            "key": key,
            "text": txt,
            "conf": conf,
            "left": int(data.get("left", [0]*n)[i]),
            "top": int(data.get("top", [0]*n)[i]),
        })
    groups = defaultdict(list)
    for r in rows:
        groups[r["key"]].append(r)
    visual_lines = []
    confs = []
    for key, items in groups.items():
        items = sorted(items, key=lambda x: x["left"])
        line = " ".join(x["text"] for x in items)
        if line.strip():
            visual_lines.append((min(x["top"] for x in items), line))
            confs.extend(x["conf"] for x in items)
    visual_lines = sorted(visual_lines, key=lambda x: x[0])
    text = "\n".join(line for _, line in visual_lines)
    conf = float(np.mean(confs) / 100.0) if confs else 0.0
    return text, conf, {"line_count": len(visual_lines), "word_count": len(rows)}

def ocr_one_image_variant(img: Image.Image, lang: str, psm: int) -> tuple[str, float, dict]:
    if not TESSERACT_AVAILABLE:
        return "", 0.0, {"error": "pytesseract_not_available"}
    config = f"--psm {psm}"
    try:
        data = pytesseract.image_to_data(img, lang=lang, config=config, output_type=pytesseract.Output.DICT)
        text, conf, meta = reconstruct_lines_from_tesseract_data(data)
        # fallback if too short
        if len(text.strip()) < 10:
            fallback = pytesseract.image_to_string(img, lang=lang, config=config)
            if len(fallback.strip()) > len(text.strip()):
                text = fallback
                meta["fallback_image_to_string"] = True
        return text, conf, meta
    except Exception as e:
        return "", 0.0, {"error": str(e), "lang": lang, "psm": psm}

def count_lab_alias_hits(text: str) -> int:
    low = (text or "").lower()
    hits = 0
    for std, alias in ALL_ALIAS_PHRASES:
        if alias.lower() in low:
            hits += 1
    return hits

def ocr_score(text: str, conf: float) -> float:
    txt = normalize_persian(text or "")
    low = txt.lower()
    med_hits = sum(1 for k in MEDICAL_KEYWORDS if normalize_persian(k).lower() in low)
    alias_hits = count_lab_alias_hits(txt)
    numeric_hits = len(re.findall(r"(?<!\d)[<>*]?\d+(?:\.\d+)?(?!\d)", txt))
    line_count = len([l for l in txt.splitlines() if l.strip()])
    return (
        min(len(txt), 4000) / 500.0
        + med_hits * 1.2
        + alias_hits * 0.8
        + min(numeric_hits, 80) * 0.08
        + min(line_count, 60) * 0.05
        + conf * 2.0
    )

def run_ocr_variants_for_image(path: Path, sid: str) -> tuple[OCRCandidate, list[OCRCandidate]]:
    candidates = []
    out_base = IMAGE_DEBUG_DIR / sid
    variants = image_variants(path, out_base)
    for variant_name, im in variants:
        for lang in CONFIG["tesseract_langs"]:
            for psm in CONFIG["psm_modes"]:
                text, conf, meta = ocr_one_image_variant(im, lang, psm)
                score = ocr_score(text, conf)
                cand = OCRCandidate(
                    text=text,
                    confidence=conf,
                    variant=variant_name,
                    psm=psm,
                    lang=lang,
                    score=score,
                    image_path=str(out_base / f"{variant_name}.png"),
                    error=meta.get("error"),
                    details=meta,
                )
                candidates.append(cand)
                # If eng+fas is missing, it often errors repeatedly; keep trying eng but don't spam too much.
    valid = [c for c in candidates if c.text and len(c.text.strip()) >= 2]
    best = max(valid or candidates, key=lambda c: c.score if c else -1)
    if CONFIG["save_variant_texts"]:
        vdir = VARIANT_TEXT_DIR / sid
        vdir.mkdir(parents=True, exist_ok=True)
        for i, c in enumerate(sorted(candidates, key=lambda x: x.score, reverse=True)[:20]):
            (vdir / f"{i:02d}_{c.variant}_psm{c.psm}_{c.lang}_score{c.score:.2f}.txt").write_text(c.text or f"ERROR: {c.error}", encoding="utf-8")
    return best, candidates



## 5) PDF text extraction and fallback OCR

For text-layer PDFs, this uses embedded text first. For scanned PDFs, it renders pages and runs the image OCR variant search.


In [6]:

# =========================
# 5) PDF extraction
# =========================

def extract_pdf_text_layer(path: Path) -> tuple[str, list[str], dict]:
    if fitz is None:
        return "", [], {"error": "pymupdf_not_available"}
    try:
        doc = fitz.open(path)
        pages = []
        for i, page in enumerate(doc[:CONFIG["max_pdf_pages"]]):
            pages.append(page.get_text() or "")
        meta = {"page_count": doc.page_count, "used_pages": len(pages), "pdf_type": "text_pdf" if any(p.strip() for p in pages) else "scanned_pdf"}
        doc.close()
        return "\f".join(pages), pages, meta
    except Exception as e:
        return "", [], {"error": str(e)}

def render_pdf_pages(path: Path, sid: str) -> list[Path]:
    if fitz is None:
        return []
    out_dir = IMAGE_DEBUG_DIR / sid / "pdf_pages"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    doc = fitz.open(path)
    for i, page in enumerate(doc[:CONFIG["max_pdf_pages"]], start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
        out = out_dir / f"page_{i:03d}.png"
        pix.save(str(out))
        paths.append(out)
    doc.close()
    return paths

def extract_text_from_file(path: Path, sid: str) -> dict:
    ext = path.suffix.lower()
    if ext == ".pdf":
        text, pages, meta = extract_pdf_text_layer(path)
        if len(text.strip()) >= CONFIG["min_ocr_text_len"]:
            return {
                "ocr_success": True,
                "text": text,
                "pages": pages,
                "ocr_confidence": 0.95,
                "source": "pdf_text_layer",
                "best_variant": "pdf_text_layer",
                "best_psm": None,
                "candidate_count": 0,
                "candidate_scores": [],
                "meta": meta,
            }
        # fallback: render and OCR pages
        rendered = render_pdf_pages(path, sid)
        page_texts = []
        best_candidates = []
        for i, img_path in enumerate(rendered, 1):
            best, cands = run_ocr_variants_for_image(img_path, f"{sid}_pdfpage{i}")
            page_texts.append(best.text)
            best_candidates.append(best)
        full_text = "\f".join(page_texts)
        avg_conf = float(np.mean([b.confidence for b in best_candidates])) if best_candidates else 0.0
        return {
            "ocr_success": bool(full_text.strip()),
            "text": full_text,
            "pages": page_texts,
            "ocr_confidence": avg_conf,
            "source": "pdf_rendered_ocr",
            "best_variant": ";".join(b.variant for b in best_candidates),
            "best_psm": ";".join(str(b.psm) for b in best_candidates),
            "candidate_count": sum(1 for _ in best_candidates),
            "candidate_scores": [asdict(b) for b in best_candidates],
            "meta": meta,
        }
    else:
        best, cands = run_ocr_variants_for_image(path, sid)
        return {
            "ocr_success": bool(best.text.strip()),
            "text": best.text,
            "pages": [best.text],
            "ocr_confidence": best.confidence,
            "source": "image_ocr_variants",
            "best_variant": best.variant,
            "best_psm": best.psm,
            "best_lang": best.lang,
            "candidate_count": len(cands),
            "candidate_scores": [
                {"variant": c.variant, "psm": c.psm, "lang": c.lang, "score": round(c.score, 2), "len": len(c.text or ""), "conf": round(c.confidence, 3), "error": c.error}
                for c in sorted(cands, key=lambda x: x.score, reverse=True)[:15]
            ],
            "meta": {},
        }



## 6) OCR text post-processing


In [7]:

# =========================
# 6) Post-process text
# =========================

def postprocess_text(text: str) -> str:
    t = normalize_persian(text or "")
    # Common OCR fixes from selected samples
    replacements = {
        "Refrence": "Reference",
        "Aeference": "Reference",
        "Ranwe": "Range",
        "Kefecence": "Reference",
        "Methad": "Method",
        "Diroid": "Thyroid",
        "Fasting Blood Glucase": "Fasting Blood Glucose",
        "Fasting blood segar": "Fasting blood sugar",
        "Hemoglobine Alc": "Hemoglobin A1c",
        "Hemoglobine A1c": "Hemoglobin A1c",
        "Cholestrol": "Cholesterol",
        "Triglycerid": "Triglycerides",
        "Creatinin": "Creatinine",
        "Tiiroid": "Thyroid",
    }
    for a, b in replacements.items():
        t = t.replace(a, b)
    # Normalize line endings and remove repeated blank lines
    t = re.sub(r"\r\n?", "\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def is_medical_text(text: str) -> tuple[bool, list[str], float]:
    t = normalize_persian(text or "")
    low = t.lower()
    hits = []
    for k in MEDICAL_KEYWORDS:
        nk = normalize_persian(k).lower()
        if nk and nk in low and k not in hits:
            hits.append(k)
    score = min(1.0, len(hits) / 4)
    return score >= 0.15, hits, score

def classify_document(text: str) -> tuple[str, float, list[str]]:
    t = normalize_persian(text or "")
    low = t.lower()
    signals = {
        "lab": ["laboratory","lab","hematology","biochemistry","serology","urine","culture","cbc","wbc","rbc","fbs","glucose","tsh","آزمایشگاه","آزمايشگاه","درمانگاه"],
        "radiology": ["radiology","ultrasound","sonography","mri","ct","x-ray","findings","impression","سونوگرافی","سونوگرافي","رادیولوژی"],
        "pap_smear": ["pap smear","hpv","nilm","ascus","asc-us","lsil","hsil","پاپ اسمیر"],
    }
    best = ("unknown_medical", 0, [])
    for doc_type, sigs in signals.items():
        found = [s for s in sigs if normalize_persian(s).lower() in low]
        if len(found) > len(best[2]):
            best = (doc_type, min(1.0, len(found)/5), found)
    return best



## 7) Common field extraction with conservative rules

This is intentionally stricter than the current repo behavior to avoid false patient names and invalid dates.


In [8]:

# =========================
# 7) Common fields
# =========================

TABLE_TOKENS = re.compile(r"\b(Result|Unit|Reference|Range|WBC|RBC|FBS|TSH|CBC|Biochemistry|Hematology|Platelet|Creatinine|Glucose)\b", re.I)

def clean_value(v: str | None, max_len: int = 120) -> str | None:
    if not v:
        return None
    v = re.sub(r"^[\s:：|#*/\\.-]+|[\s:：|#*/\\.-]+$", "", v).strip()
    v = re.sub(r"\s+", " ", v)
    if len(v) > max_len:
        return None
    return v or None

def bounded_after_label(text: str, labels: list[str], stop_words: list[str], max_chars: int = 80) -> tuple[str|None, str|None]:
    t = normalize_persian(text)
    for lab in labels:
        idx = t.find(lab)
        if idx >= 0:
            chunk = t[idx + len(lab): idx + len(lab) + max_chars]
            chunk = re.sub(r"^[\s:：,-]+", "", chunk)
            stops = [chunk.find(sw) for sw in stop_words if chunk.find(sw) >= 0]
            if stops:
                chunk = chunk[:min(stops)]
            val = clean_value(chunk, max_len=max_chars)
            if val:
                return val, t[idx: idx + len(lab) + len(chunk)]
    return None, None

def extract_common_fields(text: str) -> dict:
    t = postprocess_text(text)
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    top_text = "\n".join(lines[:12])
    full = "\n".join(lines)
    result = {}

    # Center name: top only
    center = None; center_src = None
    for i, l in enumerate(lines[:8]):
        if any(k in normalize_persian(l) for k in ["آزمایشگاه", "آزمايشگاه", "Laboratory", "درمانگاه", "بیمارستان", "Nobin", "TAV"]):
            if not TABLE_TOKENS.search(l) and len(l) <= 140:
                center = clean_value(l, 140)
                center_src = l
                break

    # National ID
    nid = None; nid_src = None
    for pat in [
        r"(?:کد\s*ملی|کد\s*ملي|كد\s*ملي|National\s*ID)\s*[:：]?\s*([0-9]{10})",
        r"\b([0-9]{10})\b",
    ]:
        m = re.search(pat, full, re.I)
        if m:
            nid = m.group(1); nid_src = m.group(0)
            break

    # Tracking/admission/report no
    tracking = None; tracking_src = None
    for pat in [
        r"\b([A-Za-z]-\d{5}-\d{3,})\b",
        r"(?:پذیرش|پذيرش|Admission|Report\s*No|شماره)\s*[:：]?\s*([A-Za-z0-9_-]{3,})",
    ]:
        m = re.search(pat, full, re.I)
        if m:
            tracking = m.group(1); tracking_src = m.group(0); break

    # Date: prefer labelled and validate Jalali
    date = None; date_src = None
    labelled_date_patterns = [
        r"(?:تاریخ\s*پذیرش|تاريخ\s*پذيرش|تاریخ\s*جواب|تاریخ\s*گزارش|Report\s*Date|Date)\s*[:：]?\s*(?:\d{1,2}:\d{2}:\d{2}\s*[-–]?\s*)?([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
        r"([0-9]{4}[/-][0-9]{1,2}[/-][0-9]{1,2})",
    ]
    for pat in labelled_date_patterns:
        for m in re.finditer(pat, full, re.I):
            cand = normalize_digits(m.group(1))
            if validate_jalali_date(cand):
                date = cand; date_src = m.group(0); break
        if date:
            break

    # Patient line / name / sex / age
    patient_name = None; patient_src = None; sex = None; sex_src = None; age = None; age_src = None

    # TAV style after NFKC normalization:
    # "امینی تهران- آقای کورش- دکتر34 : سن"
    # "روشن- خانم مهلا- دکتر27 : سن"
    tav_pat = re.search(
        r"([^\n:]{1,70}?)-\s*(آقای|آقاي|خانم)\s+([^\n:]{1,45}?)-\s*د(?:ك|ک)تر\s*([0-9]{1,3})\s*[:：]?\s*سن",
        full
    )
    if tav_pat:
        family = clean_value(tav_pat.group(1), 55)
        given = clean_value(tav_pat.group(3), 45)
        if family and given and not TABLE_TOKENS.search(family + " " + given):
            patient_name = clean_value(f"{given} {family}", 80)
            patient_src = tav_pat.group(0)
        sex = "female" if tav_pat.group(2) == "خانم" else "male"
        sex_src = tav_pat.group(0)
        age_candidate = int(tav_pat.group(4))
        if 0 <= age_candidate <= 120:
            age = age_candidate
            age_src = tav_pat.group(0)

    if not patient_name:
        # Persian labels. This is bounded to avoid taking an entire OCR paragraph.
        val, src = bounded_after_label(
            full,
            ["نام بیمار", "نام بيمار", "نام مراجعه کننده", "نام مراجعـه کننده", "Name:"],
            ["تاریخ", "تاريخ", "سن", "شماره", "پزشک", "پزشك", "کد", "كد", "Sex", "Age", "Date", "Ward", "Comments", "Result", "Unit"],
            max_chars=80
        )
        if val and not TABLE_TOKENS.search(val) and len(re.findall(r"\d", val)) <= 2:
            # Remove common leading separators and titles, but keep خانم/آقای context for sex.
            patient_name = clean_value(re.sub(r"^(خانم|آقای|آقاي)\s+", "", val), 80)
            patient_src = src
            if "خانم" in val:
                sex = sex or "female"; sex_src = sex_src or src
            elif "آقای" in val or "آقاي" in val:
                sex = sex or "male"; sex_src = sex_src or src

    if not sex:
        # only use patient/name context, not doctor line
        context = patient_src or ""
        if "خانم" in context or re.search(r"\bFemale\b", context, re.I):
            sex = "female"; sex_src = context
        elif "آقای" in context or "آقاي" in context or re.search(r"\bMale\b", context, re.I):
            sex = "male"; sex_src = context
        else:
            # English analyzer reports often have "Sex: Female" in top header
            msex = re.search(r"Sex\s*[:：]?\s*(Female|Male)", top_text, re.I)
            if msex:
                sex = msex.group(1).lower()
                sex_src = msex.group(0)

    # Age: only explicit patient age and valid range.
    # Also support TAV reversed style "دکتر34 : سن" already handled above.
    if age is None:
        for pat in [
            r"(?:سن\s*[:：]?\s*|Age\s*[:：]?\s*)([0-9]{1,3})(?:\s*سال)?",
            r"([0-9]{1,3})\s*(?:سال)\b",
        ]:
            for m in re.finditer(pat, top_text, re.I):
                src = m.group(0)
                if re.search(r"Below age|Above age|reference|Adults|Children|years?:", src, re.I):
                    continue
                a = int(m.group(1))
                if 0 <= a <= 120:
                    age = a; age_src = src; break
            if age is not None:
                break

    # Doctor: strict short label
    doctor = None; doctor_src = None
    for pat in [
        r"(?:پزشک\s*معالج|پزشك\s*معالج|پزشک|پزشك|Doctor|Physician)\s*[:：]?\s*([^\n]{2,80})",
    ]:
        m = re.search(pat, top_text, re.I)
        if m:
            cand = clean_value(m.group(1), 80)
            if cand and not TABLE_TOKENS.search(cand):
                doctor, doctor_src = cand, m.group(0)
                break

    def f(value, confidence=0.85, source=None, extra=None):
        out = {"value": value, "confidence": confidence if value not in [None, ""] else 0.0, "source_text": source}
        if extra: out.update(extra)
        return out

    result["center_name"] = f(center, 0.7, center_src)
    result["national_id"] = {"value": None, "hash": hash_value(nid), "confidence": 0.9 if nid else 0.0, "source_text": nid_src.replace(nid, "*"*10) if nid and nid_src else None}
    result["tracking_number"] = f(tracking, 0.85, tracking_src)
    result["date_of_test_or_report"] = f(date, 0.9, date_src, {"calendar": "jalali" if date else None})
    result["patient_name"] = f(patient_name, 0.85 if patient_name else 0.0, patient_src)
    result["sex"] = f(sex, 0.85 if sex else 0.0, sex_src)
    result["age"] = f(age, 0.85 if age is not None else 0.0, age_src)
    result["doctor_name"] = f(doctor, 0.75 if doctor else 0.0, doctor_src)
    return result



## 8) Lab, urine, culture, and radiology extraction

This section saves each extracted part/test/field in its own row.


In [9]:

# =========================
# 8) Structured extraction
# =========================

NUM_RE = re.compile(r"(?<!\d)([*<>]?\d+(?:\.\d+)?)(?!\d)")
UNIT_RE = re.compile(r"(10\^?\s*[36]\s*/?\s*(?:uL|µL|μL)|10\*?[36]\s*/?\s*(?:uL|µL|μL)|mg/dL|mg/dl|g/dL|g/dl|U/L|IU/L|mIU/L|µIU/mL|uIU/mL|ng/mL|ng/ml|pg/ml|IU/L|%|fL|fl|pg|Ratio|Sec|mm/hr)", re.I)
FLAG_RE = re.compile(r"\b(High|Low|H|L|Critical|\*)\b", re.I)

GUIDELINE_WORDS = re.compile(r"\b(Normal|Borderline|Desirable|High risk|Low risk|Adults|Male|Female|Women|Men|Pregnant|Below age|Above age|Reference|Range)\b", re.I)

def alias_pattern(alias: str) -> str:
    return re.escape(alias).replace(r"\ ", r"\s+")

def extract_unit(window: str) -> str | None:
    m = UNIT_RE.search(window)
    return m.group(0) if m else None

def extract_flag(window: str) -> str | None:
    m = FLAG_RE.search(window)
    if not m:
        return None
    val = m.group(1)
    if val == "*":
        return "rechecked"
    return val

def plausible_numeric_result(std: str, alias: str, window: str) -> tuple[str|None, float|None]:
    # Remove alias part at beginning and find first numeric after alias.
    cleaned = window
    # Prefer numeric immediately after alias or within first 80 chars
    nums = []
    for m in NUM_RE.finditer(cleaned[:120]):
        val = m.group(1)
        start = m.start()
        # skip years/dates, reference ranges with many digits, phone numbers
        if re.search(r"\d{4}[/-]\d", cleaned[max(0, start-8):start+20]):
            continue
        # skip values clearly after guideline words before them
        before = cleaned[max(0, start-45):start]
        if GUIDELINE_WORDS.search(before) and start > 40:
            continue
        try:
            numeric = float(re.sub(r"^[*<>]+", "", val))
        except Exception:
            numeric = None
        nums.append((start, val, numeric))
    if not nums:
        return None, None
    nums = sorted(nums, key=lambda x: x[0])
    return nums[0][1].lstrip("*"), nums[0][2]

def extract_lab_numeric(text: str) -> list[dict]:
    t = postprocess_text(text)
    rows = []
    flat = re.sub(r"\s+", " ", t)
    line_joined = "\n".join([l.strip() for l in t.splitlines() if l.strip()])
    search_spaces = [line_joined, flat]
    for std, alias in ALL_ALIAS_PHRASES:
        # avoid overly generic Glucose in urine contexts: still useful but handled separately
        pat = re.compile(rf"(?i)(?<![A-Za-z]){alias_pattern(alias)}(?![A-Za-z])")
        for space_name, src_text in [("lines", line_joined), ("flat", flat)]:
            for m in pat.finditer(src_text):
                start = m.start()
                window = src_text[start:start+180]
                # reject table header-only windows
                result, numeric = plausible_numeric_result(std, alias, window)
                if result is None:
                    continue
                unit = extract_unit(window)
                flag = extract_flag(window)
                # Reference range: keep short reference-like fragment after unit/result if visible
                ref = None
                ref_m = re.search(r"([<>]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?)", window[window.find(result)+len(result):])
                if ref_m:
                    ref = ref_m.group(1).replace(" ", "")
                rows.append({
                    "part_type": "lab_result",
                    "section": infer_section(std, t),
                    "field_name": std,
                    "test_name_raw": alias,
                    "value": result,
                    "result_numeric": numeric,
                    "unit": unit,
                    "reference_range": ref,
                    "flag": flag,
                    "method": infer_method(window),
                    "confidence": 0.65 if space_name=="lines" else 0.50,
                    "source_text": window[:250],
                })
                break
            # If found one per alias/std, continue to next alias; dedup later
    return deduplicate_rows(rows)

def infer_method(window: str) -> str | None:
    for m in ["Photometr", "ELISA", "ECLIA", "CLIA", "EIA", "EIA & ELFA", "HPLC", "FLC"]:
        if m.lower() in window.lower():
            return m
    return None

def infer_section(std: str, text: str) -> str:
    if std in {"WBC","RBC","HGB","HCT","MCV","MCH","MCHC","RDW-CV","RDW-SD","PLT","PDW","MPV","NEUT#","LYMPH#","MONO#","EO#","BASO#","NEUT%","LYMPH%","MONO%","EO%","BASO%","ESR"}:
        return "hematology"
    if std in {"PT","PT Control","INR","PTT"}:
        return "coagulation"
    if std in {"TSH","T3","T4","Free T4","Vitamin D","Vitamin B12","Ferritin","CRP","LH","FSH","Prolactin","Testosterone","Free Testosterone","Estradiol","DHEA-SO4","17OH-Progesterone"}:
        return "special_biochemistry_or_hormone"
    if std in {"Urine Culture","Color","Appearance","Specific Gravity","pH","Protein","Urine Glucose","Ketone","Bilirubin Urine","Urobilinogen","Nitrite","Blood/Hb","WBC/HPF","RBC/HPF","Epithelial Cells","Bacteria","Mucus","Casts","Crystals"}:
        return "urine_or_microbiology"
    return "biochemistry"

def deduplicate_rows(rows: list[dict]) -> list[dict]:
    best = {}
    for r in rows:
        key = (r.get("part_type"), r.get("field_name"), str(r.get("value")).lower())
        if key not in best or r.get("confidence",0) > best[key].get("confidence",0):
            best[key] = r
    return list(best.values())

QUALITATIVE_TERMS = ["Negative", "Positive", "Trace", "Normal", "Yellow", "Clear", "Acid", "Not seen", "Few", "Rare", "Moderate", "Many", "No growth after 24 hrs", "No growth after 48 hrs", "No bacteria growth after 48 hrs"]

def extract_urine_and_culture(text: str) -> list[dict]:
    t = postprocess_text(text)
    flat = re.sub(r"\s+", " ", t)
    rows = []
    # Culture
    culture_m = re.search(r"(No\s+(?:bacteria\s+)?growth\s+after\s+(?:24|48)\s*hrs?\.?)", flat, re.I)
    if culture_m:
        rows.append({
            "part_type": "lab_result",
            "section": "microbiology",
            "field_name": "Urine Culture",
            "test_name_raw": "Culture",
            "value": culture_m.group(1),
            "result_numeric": None,
            "unit": None,
            "reference_range": None,
            "flag": None,
            "method": None,
            "confidence": 0.85,
            "source_text": flat[max(0, culture_m.start()-80):culture_m.end()+80],
        })
    # Urine qualitative tokens
    urine_items = {
        "Color": r"Color\s+(Yellow|Amber|Red|Pale|Straw)",
        "Appearance": r"Appearance\s+(Clear|Semi[- ]?Clear|Turbid|Cloudy)",
        "Specific Gravity": r"(?:Specific Gravity|Sp\.?\s*Gravity)\s+([0-9.]+)",
        "pH": r"\bPH\b|\bpH\b\s*([0-9.]+|Acid|Alkaline)",
        "Protein": r"Protein\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urine Glucose": r"Glucose\s+(Negative|Positive|Trace|[0-9+]+)",
        "Ketone": r"Ketone\s+(Negative|Positive|Trace|[0-9+]+)",
        "Bilirubin Urine": r"Bilirubin\s+(Negative|Positive|Trace|[0-9+]+)",
        "Urobilinogen": r"Urobilinogen\s+(Normal|Negative|Positive|[0-9.]+)",
        "Nitrite": r"Nitrite\s+(Negative|Positive)",
        "Blood/Hb": r"(?:Blood/Hb|Hemoglobin|Blood)\s+(Negative|Positive|Trace|[0-9+]+)",
        "WBC/HPF": r"(?:W\.?B\.?C\.?\s*/?\s*HPF|WBC/HPF|WBC)\s+([0-9]+\s*[-–]\s*[0-9]+|[0-9]+|Few|Rare|Many)",
        "RBC/HPF": r"(?:R\.?B\.?C\.?\s*/?\s*HPF|RBC/HPF|RBC)\s+([0-9]+\s*[-–]\s*[0-9]+|[0-9]+|Few|Rare|Many)",
        "Bacteria": r"Bacteria\s+(Negative|Not seen|Look at culture|Few|Rare|Many)",
        "Mucus": r"Mucus\s+(Negative|Not seen|Few|Rare|Many)",
        "Casts": r"Casts?\s+(Negative|Not seen|Few|Rare|Many)",
        "Crystals": r"Crystals?\s+(Negative|Not seen|Few|Rare|Many)",
    }
    for name, pat in urine_items.items():
        m = re.search(pat, flat, re.I)
        if m:
            val = m.group(1) if m.groups() else m.group(0)
            rows.append({
                "part_type": "lab_result",
                "section": "urine_analysis",
                "field_name": name,
                "test_name_raw": name,
                "value": val,
                "result_numeric": None,
                "unit": None,
                "reference_range": None,
                "flag": None,
                "method": None,
                "confidence": 0.65,
                "source_text": flat[max(0, m.start()-60):m.end()+60],
            })
    return deduplicate_rows(rows)

def extract_radiology(text: str) -> list[dict]:
    t = postprocess_text(text)
    low = t.lower()
    rows = []
    if any(k in low for k in ["ultrasound", "sonography"]) or "سونوگرافی" in t or "سونوگرافي" in t:
        rows.append({"part_type": "radiology", "section": "radiology", "field_name": "imaging_modality", "value": "ultrasound", "confidence": 0.8, "source_text": None})
    if "شکم" in t and "لگن" in t:
        rows.append({"part_type": "radiology", "section": "radiology", "field_name": "body_part_or_exam_name", "value": "abdomen and pelvis", "confidence": 0.75, "source_text": None})
    if len(t) > 30 and (rows or "Findings" in t or "Impression" in t or "سونو" in t):
        rows.append({"part_type": "radiology", "section": "radiology", "field_name": "full_narrative_report", "value": t[:4000], "confidence": 0.65, "source_text": t[:300]})
    return rows

def extract_all_parts(text: str, document_type: str) -> list[dict]:
    rows = []
    common = extract_common_fields(text)
    for k, v in common.items():
        value = v.get("hash") if k == "national_id" else v.get("value")
        if value is not None:
            rows.append({
                "part_type": "common_field",
                "section": "header",
                "field_name": k,
                "value": value,
                "result_numeric": None,
                "unit": None,
                "reference_range": None,
                "flag": None,
                "method": None,
                "confidence": v.get("confidence", 0),
                "source_text": v.get("source_text"),
            })
    lab_rows = extract_lab_numeric(text) + extract_urine_and_culture(text)
    rows.extend(deduplicate_rows(lab_rows))
    if document_type == "radiology":
        rows.extend(extract_radiology(text))
    return rows



## 9) Process one file interactively


In [10]:

# =========================
# 9) One-file test
# =========================

SAMPLE_INDEX = 0  # Change this to test another file
sample_path = FILES[SAMPLE_INDEX]
sid = safe_id(sample_path)

print("Testing:", SAMPLE_INDEX, sample_path.name)
print("Path:", sample_path)

quality = assess_file_quality(sample_path)
ocr_info = extract_text_from_file(sample_path, sid)
text_raw = ocr_info["text"]
text = postprocess_text(text_raw)
medical_ok, med_hits, rel_score = is_medical_text(text)
doc_type, doc_conf, doc_signals = classify_document(text)
parts = extract_all_parts(text, doc_type)

print("\nQUALITY")
print(json.dumps(quality, ensure_ascii=False, indent=2))

print("\nOCR")
print("success:", ocr_info["ocr_success"])
print("source:", ocr_info["source"])
print("confidence:", ocr_info["ocr_confidence"])
print("text length:", len(text))
print("best_variant:", ocr_info.get("best_variant"))
print("best_psm:", ocr_info.get("best_psm"))

print("\nRELEVANCE / CLASSIFICATION")
print("medical_ok:", medical_ok, "rel_score:", rel_score, "hits:", med_hits[:20])
print("document_type:", doc_type, "confidence:", doc_conf, "signals:", doc_signals)

print("\nCOMMON FIELDS")
common = extract_common_fields(text)
for k, v in common.items():
    print(k, "=>", v.get("value") or v.get("hash"), "| conf:", v.get("confidence"), "| src:", v.get("source_text"))

print("\nEXTRACTED PARTS (first 50)")
parts_df = pd.DataFrame(parts)
display(parts_df.head(50) if not parts_df.empty else parts_df)

print("\nTEXT PREVIEW")
print(text[:3000])

# Save one-file outputs
(TEXT_DIR / f"{sid}.txt").write_text(text, encoding="utf-8")
with open(OUTPUT_DIR / f"{sid}.json", "w", encoding="utf-8") as f:
    json.dump({
        "filename": sample_path.name,
        "quality": quality,
        "ocr": {k:v for k,v in ocr_info.items() if k not in ["text", "pages"]},
        "medical_ok": medical_ok,
        "relevance_score": rel_score,
        "document_type": doc_type,
        "common_fields": common,
        "parts": parts,
        "text_path": str(TEXT_DIR / f"{sid}.txt"),
    }, f, ensure_ascii=False, indent=2)

print("\nSaved:", TEXT_DIR / f"{sid}.txt")


Testing: 0 0014161672_14041209_O_00404121721.pdf
Path: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/samples/selected samples/0014161672_14041209_O_00404121721.pdf

QUALITY
{
  "status": "pdf_quality_not_assessed_here",
  "overall_quality_score": null,
  "issues": [],
  "width": null,
  "height": null
}

OCR
success: True
source: pdf_text_layer
confidence: 0.95
text length: 960
best_variant: pdf_text_layer
best_psm: None

RELEVANCE / CLASSIFICATION
medical_ok: True rel_score: 1.0 hits: ['hematology', 'biochemistry', 'hemoglobin', 'platelet', 'pt', 'آزمایشگاه', 'آزمايشگاه', 'پاتوبیولوژی', 'پاتوبيولوژی', 'پزشک', 'پذیرش']
document_type: lab confidence: 0.8 signals: ['hematology', 'biochemistry', 'آزمایشگاه', 'آزمايشگاه']

COMMON FIELDS
center_name => آزمایشگاه پاتوبیولوژی تاو | conf: 0.7 | src: # آزمایشگاه پاتوبیولوژی تاو#
national_id => 6f7b8e1c3f733b32eaf2a037ee257d451f9423ceb96b2e95426f83929403d282 | conf: 0.9 | src: کد ملی :****

,part_type,section,field_name,value,result_numeric,unit,reference_range,flag,method,confidence,source_text,test_name_raw
0,common_field,header,center_name,آزمایشگاه پاتوبیولوژی تاو,NaN,NaN,NaN,NaN,NaN,0.70,# آزمایشگاه پاتوبیولوژی تاو#,NaN
1,common_field,header,national_id,6f7b8e1c3f733b32eaf2a037ee257d451f9423ceb96b2e...,NaN,NaN,NaN,NaN,NaN,0.90,کد ملی :**********,NaN
2,common_field,header,tracking_number,O-40412-1721,NaN,NaN,NaN,NaN,NaN,0.85,O-40412-1721,NaN
3,common_field,header,date_of_test_or_report,1404/12/09,NaN,NaN,NaN,NaN,NaN,0.90,تاریخ پذیرش1404/12/09,NaN
4,common_field,header,patient_name,کورش امینی طھران,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,NaN
5,common_field,header,sex,male,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,NaN
6,common_field,header,age,34,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,NaN
7,common_field,header,doctor_name,امینی طھران- آقای کورش- دکتر34 : سن,NaN,NaN,NaN,NaN,NaN,0.75,پزشک\nامینی طھران- آقای کورش- دکتر34 : سن,NaN
8,lab_result,hematology,WBC,7.24,7.24,10^3/uL,3.5-10.5,H,NaN,0.65,W.B.C\n7.24\n10^3/uL\n3.5-10.5\nR.B.C\n4.9\n10...,W.B.C
9,lab_result,hematology,RBC,4.9,4.90,10^6/uL,4.32-5.72,H,NaN,0.65,R.B.C\n4.9\n10^6/uL\n4.32 - 5.72\nHemoglobin\n...,R.B.C



TEXT PREVIEW
# آزمایشگاه پاتوبیولوژی تاو#
 تھران، شھرک غرب، بلوار دریا،نبش چھارراه قدس، پلاک20
021-91012828
O-40412-1721
: پزشک
 امینی طھران- آقای کورش- دکتر34 : سن
شماره :
نام :
کد ملی :0014161672
 - سالMale
: تاریخ پذیرش1404/12/09 - 14:11:34
Test
Result
Hematology
Unit
Normal Range
Flag
Method
W.B.C
7.24
10^3/uL
3.5-10.5
R.B.C
4.9
10^6/uL
4.32 - 5.72
Hemoglobin
15.0
g/dL
13.5 - 17.5
H.C.T
45.0
%
38.8-50 %
M.C.V
91.8
fL
81.2-95.1 fl
M.C.H
30.6
pg
25.8-33.1 pg
M.C.H.C
33.3
g/dL
32-36 g/dl
Platelets
265
10^3 /uL
150 - 450
RDW-CV
12.6
%
11.8-15.6 %
RDW-SD
40.7
fl
36 - 54
PDW
14.3
fL
9-17 fl
Neutrophils
49.7
%
40 - 80
Lymphocytes
43
%
20 - 40
High
Monocytes
5.5
%
4.1-12.4 %
Eosinophils
1.7
%
0.4-6.7 %
Basophils
0.1
%
0.3-1.4 %
Test
Result
Biochemistry
Unit
Normal Range
Flag
Method
Fasting blood sugar
91
mg/dL
70-115
Photometr
SGOT(AST)
19
U/L
<37
Photometr
SGPT(ALT)
21
U/L
< 40
Photometr
This answer is signed out by Dr.N.Adanvar
Print On 1405/01/30-13:48:08 By 2

Saved: /Users/aliebrahim


## 10) Batch process all selected samples — JSON-first outputs

This version saves **both table-friendly CSVs and JSON-friendly outputs**.

Outputs:

1. `document_summary.csv` — flat dashboard for quick review.
2. `document_json_rows.csv` — **one row per file**, with nested JSON columns:
   - `quality_json`
   - `ocr_json`
   - `common_fields_json`
   - `extracted_parts_json`
   - `lab_results_json`
   - `document_json`
   - `extracted_text`
3. `documents.jsonl` — one complete JSON object per file, ideal for backend/product review.
4. `extracted_parts.csv` — one row per field/test/result, good for filtering and QA.
5. `extracted_texts.csv` — exact OCR/PDF extracted text per file.
6. `texts/*.txt` — exact text saved as a standalone file per document.
7. `*.json` — complete JSON file per document.

Use `document_json_rows.csv` when you want “CSV but JSON-like for each file”.


In [11]:

# =========================
# 10) Batch — JSON-first output
# =========================

def decide_status(path: Path, quality: dict, ocr_info: dict, text: str, medical_ok: bool, doc_type: str, parts: list[dict]) -> tuple[str, list[str]]:
    warnings = []
    if path.suffix.lower() not in SUPPORTED_EXTS:
        return "invalid_file", ["unsupported_extension"]
    if quality.get("status") in ["poor_quality", "unreadable"] and path.suffix.lower() != ".pdf":
        return "poor_quality_rejected", quality.get("issues", [])
    if not ocr_info.get("ocr_success") or len((text or "").strip()) < CONFIG["min_ocr_text_len"]:
        return "ocr_failed", ["ocr_text_too_short"]
    if not medical_ok:
        return "unrelated_or_not_medical", ["no_medical_signals"]

    lab_count = sum(1 for r in parts if r.get("part_type") == "lab_result")
    common_count = sum(1 for r in parts if r.get("part_type") == "common_field")
    radiology_count = sum(1 for r in parts if r.get("part_type") == "radiology")

    if doc_type == "lab" and lab_count == 0:
        return "partial_extraction", ["no_lab_rows_extracted"]

    # Make status more honest:
    # Good extraction requires enough structured information, not only lab rows.
    if doc_type == "lab":
        if lab_count >= 3 and common_count >= 2:
            return "extracted_good", warnings
        if lab_count >= 3:
            return "needs_manual_review", ["few_common_fields"]
        return "needs_manual_review", ["low_structured_data"]

    if doc_type == "radiology":
        if radiology_count >= 2:
            return "extracted_good", warnings
        return "needs_manual_review", ["low_radiology_structure"]

    return "needs_manual_review", warnings or ["low_structured_data"]


def make_document_json(
    path: Path,
    index: int,
    sid: str,
    quality: dict,
    ocr_info: dict,
    text: str,
    medical_ok: bool,
    med_hits: list[str],
    rel_score: float,
    doc_type: str,
    doc_conf: float,
    doc_signals: list[str],
    parts: list[dict],
    status: str,
    warnings: list[str],
    text_path: Path,
) -> dict:
    common_fields = {}
    lab_results = []
    radiology = {}
    other_parts = []

    for r in parts:
        if r.get("part_type") == "common_field":
            common_fields[r.get("field_name")] = {
                "value": r.get("value"),
                "confidence": r.get("confidence"),
                "source_text": r.get("source_text"),
            }
        elif r.get("part_type") == "lab_result":
            lab_results.append({
                "section": r.get("section"),
                "test_name_standard": r.get("field_name"),
                "test_name_raw": r.get("test_name_raw"),
                "result_value": r.get("value"),
                "result_numeric": r.get("result_numeric"),
                "unit": r.get("unit"),
                "reference_range": r.get("reference_range"),
                "flag": r.get("flag"),
                "method": r.get("method"),
                "confidence": r.get("confidence"),
                "source_text": r.get("source_text"),
            })
        elif r.get("part_type") == "radiology":
            radiology[r.get("field_name")] = {
                "value": r.get("value"),
                "confidence": r.get("confidence"),
                "source_text": r.get("source_text"),
            }
        else:
            other_parts.append(r)

    return {
        "document": {
            "index": index,
            "filename": path.name,
            "file_path": str(path),
            "file_ext": path.suffix.lower(),
            "mime_type": file_mime_guess(path),
            "file_size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
            "status": status,
            "warnings": warnings,
            "document_type": doc_type,
            "document_type_confidence": doc_conf,
            "medical_ok": medical_ok,
            "relevance_score": rel_score,
            "medical_hits": med_hits[:30],
            "document_signals": doc_signals,
        },
        "quality": quality,
        "ocr": {
            "success": ocr_info.get("ocr_success"),
            "confidence": ocr_info.get("ocr_confidence"),
            "text_length": len(text),
            "source": ocr_info.get("source"),
            "best_variant": ocr_info.get("best_variant"),
            "best_psm": ocr_info.get("best_psm"),
            "best_lang": ocr_info.get("best_lang"),
            "candidate_count": ocr_info.get("candidate_count"),
            "candidate_scores": ocr_info.get("candidate_scores"),
            "text_path": str(text_path),
        },
        "common_fields": common_fields,
        "extracted_data": {
            "lab_results": lab_results,
            "radiology": radiology if radiology else None,
            "other_parts": other_parts,
        },
        "extracted_text": text,
    }


def process_one_file(path: Path, index: int) -> tuple[dict, list[dict], dict]:
    sid = f"{index:04d}_{safe_id(path)}"
    base_record = {
        "index": index,
        "filename": path.name,
        "file_path": str(path),
        "file_ext": path.suffix.lower(),
        "mime_type": file_mime_guess(path),
        "file_size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
    }

    try:
        quality = assess_file_quality(path)
        ocr_info = extract_text_from_file(path, sid)
        text = postprocess_text(ocr_info.get("text", ""))
        medical_ok, med_hits, rel_score = is_medical_text(text)
        doc_type, doc_conf, doc_signals = classify_document(text)
        parts = extract_all_parts(text, doc_type)
        status, warnings = decide_status(path, quality, ocr_info, text, medical_ok, doc_type, parts)

        text_path = TEXT_DIR / f"{sid}.txt"
        text_path.write_text(text, encoding="utf-8")

        document_json = make_document_json(
            path=path,
            index=index,
            sid=sid,
            quality=quality,
            ocr_info=ocr_info,
            text=text,
            medical_ok=medical_ok,
            med_hits=med_hits,
            rel_score=rel_score,
            doc_type=doc_type,
            doc_conf=doc_conf,
            doc_signals=doc_signals,
            parts=parts,
            status=status,
            warnings=warnings,
            text_path=text_path,
        )

        json_path = OUTPUT_DIR / f"{sid}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(document_json, f, ensure_ascii=False, indent=2)

        for r in parts:
            r.update({
                "index": index,
                "filename": path.name,
                "document_status": status,
                "document_type": doc_type,
                "text_path": str(text_path),
                "json_path": str(json_path),
            })

        lab_count = sum(1 for r in parts if r.get("part_type") == "lab_result")
        common_count = sum(1 for r in parts if r.get("part_type") == "common_field")
        radiology_count = sum(1 for r in parts if r.get("part_type") == "radiology")

        record = dict(base_record)
        record.update({
            "status": status,
            "warnings": ";".join(warnings),
            "quality_status": quality.get("status"),
            "quality_score": quality.get("overall_quality_score"),
            "quality_issues": ";".join(quality.get("issues", [])),
            "width": quality.get("width"),
            "height": quality.get("height"),
            "ocr_success": ocr_info.get("ocr_success"),
            "ocr_confidence": round(float(ocr_info.get("ocr_confidence") or 0), 3),
            "ocr_text_length": len(text),
            "ocr_source": ocr_info.get("source"),
            "best_ocr_variant": ocr_info.get("best_variant"),
            "best_psm": ocr_info.get("best_psm"),
            "medical_ok": medical_ok,
            "relevance_score": rel_score,
            "medical_hits": ";".join(med_hits[:20]),
            "document_type": doc_type,
            "document_type_confidence": doc_conf,
            "document_signals": ";".join(doc_signals),
            "common_field_count": common_count,
            "lab_result_count": lab_count,
            "radiology_field_count": radiology_count,
            "patient_name_found": "patient_name" in document_json["common_fields"],
            "date_found": "date_of_test_or_report" in document_json["common_fields"],
            "age_found": "age" in document_json["common_fields"],
            "sex_found": "sex" in document_json["common_fields"],
            "text_path": str(text_path),
            "json_path": str(json_path),
        })

        return record, parts, document_json

    except Exception as e:
        err_json = {
            "document": {**base_record, "status": "notebook_error", "warnings": [str(e)], "document_type": "unknown"},
            "quality": None,
            "ocr": {"success": False, "text_length": 0},
            "common_fields": {},
            "extracted_data": {"lab_results": [], "radiology": None, "other_parts": []},
            "extracted_text": "",
            "error_trace": traceback.format_exc(),
        }
        record = dict(base_record)
        record.update({
            "status": "notebook_error",
            "warnings": str(e),
            "quality_status": None,
            "ocr_success": False,
            "ocr_text_length": 0,
            "document_type": "unknown",
            "lab_result_count": 0,
            "json_path": None,
        })
        return record, [], err_json


summary_records = []
part_records = []
document_json_records = []
text_records = []
json_csv_rows = []

for i, p in enumerate(FILES, start=1):
    print(f"[{i}/{len(FILES)}] {p.name}")
    rec, parts, doc_json = process_one_file(p, i)
    summary_records.append(rec)
    part_records.extend(parts)
    document_json_records.append(doc_json)

    text_records.append({
        "index": i,
        "filename": p.name,
        "status": doc_json.get("document", {}).get("status"),
        "document_type": doc_json.get("document", {}).get("document_type"),
        "ocr_text_length": len(doc_json.get("extracted_text", "")),
        "text_path": rec.get("text_path"),
        "extracted_text": doc_json.get("extracted_text", ""),
    })

    json_csv_rows.append({
        "index": i,
        "filename": p.name,
        "status": doc_json.get("document", {}).get("status"),
        "document_type": doc_json.get("document", {}).get("document_type"),
        "quality_json": json.dumps(doc_json.get("quality"), ensure_ascii=False),
        "ocr_json": json.dumps(doc_json.get("ocr"), ensure_ascii=False),
        "common_fields_json": json.dumps(doc_json.get("common_fields"), ensure_ascii=False),
        "lab_results_json": json.dumps(doc_json.get("extracted_data", {}).get("lab_results", []), ensure_ascii=False),
        "extracted_parts_json": json.dumps(parts, ensure_ascii=False),
        "document_json": json.dumps(doc_json, ensure_ascii=False),
        "extracted_text": doc_json.get("extracted_text", ""),
        "json_path": rec.get("json_path"),
        "text_path": rec.get("text_path"),
    })


summary_df = pd.DataFrame(summary_records)
parts_df = pd.DataFrame(part_records)
texts_df = pd.DataFrame(text_records)
json_rows_df = pd.DataFrame(json_csv_rows)

summary_path = OUTPUT_DIR / "document_summary.csv"
parts_path = OUTPUT_DIR / "extracted_parts.csv"
texts_path = OUTPUT_DIR / "extracted_texts.csv"
json_rows_path = OUTPUT_DIR / "document_json_rows.csv"
jsonl_path = OUTPUT_DIR / "documents.jsonl"

summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
parts_df.to_csv(parts_path, index=False, encoding="utf-8-sig")
texts_df.to_csv(texts_path, index=False, encoding="utf-8-sig")
json_rows_df.to_csv(json_rows_path, index=False, encoding="utf-8-sig")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for obj in document_json_records:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("\nSaved summary:", summary_path)
print("Saved parts:", parts_path)
print("Saved exact texts CSV:", texts_path)
print("Saved JSON rows CSV:", json_rows_path)
print("Saved JSONL:", jsonl_path)

print("\nStatus counts:")
display(summary_df["status"].value_counts().to_frame("count"))

print("\nDocument type counts:")
display(summary_df["document_type"].value_counts().to_frame("count"))

display(summary_df.head(20))
display(parts_df.head(50))
display(json_rows_df[["index", "filename", "status", "document_type", "json_path", "text_path"]].head(20))


[1/20] 0014161672_14041209_O_00404121721.pdf
[2/20] 0021858845_14041209_O_00404121731.pdf
[3/20] 0023471026_14041209_O_00404121728.pdf
[4/20] 0025631314_14041209_O_00404121726.pdf
[5/20] 0024150010_14041209_O_00404121722.pdf
[6/20] 0025692283_14041209_O_00404121730.pdf
[7/20] 0020139871_14041209_O_00404121714.pdf
[8/20] 20260427_181919.jpg
[9/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg
[10/20] ۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg
[11/20] 20260427_181713.jpg
[12/20] -2147483648_-210195.jpg
[13/20] -2147483648_-210193.jpg
[14/20] ۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg
[15/20] 20260427_181636.jpg
[16/20] 20260427_181554.jpg
[17/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۳۶.jpg
[18/20] ۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg
[19/20] 20260427_181654.jpg
[20/20] ۲۰۲۶۰۴۲۸_۱۵۰۱۲۰.jpg

Saved summary: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selected_samples_v5/document_summary.csv
Saved parts: /Users/aliebrahimi/Documents/Saman Salamat/Extract text from files/Extract-text-from-medical-documents/notebook_outputs/selec

,count
status,
extracted_good,12
needs_manual_review,8



Document type counts:


,count
document_type,
lab,19
unknown_medical,1


,index,filename,file_path,file_ext,mime_type,file_size_kb,status,warnings,quality_status,quality_score,...,document_signals,common_field_count,lab_result_count,radiology_field_count,patient_name_found,date_found,age_found,sex_found,text_path,json_path
0,1,0014161672_14041209_O_00404121721.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,44.22,extracted_good,,pdf_quality_not_assessed_here,NaN,...,hematology;biochemistry;آزمایشگاه;آزمايشگاه,8,22,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
1,2,0021858845_14041209_O_00404121731.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,42.88,extracted_good,,pdf_quality_not_assessed_here,NaN,...,hematology;آزمایشگاه;آزمايشگاه,8,17,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
2,3,0023471026_14041209_O_00404121728.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,40.79,needs_manual_review,low_structured_data,pdf_quality_not_assessed_here,NaN,...,biochemistry;آزمایشگاه;آزمايشگاه,8,2,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
3,4,0025631314_14041209_O_00404121726.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,41.18,extracted_good,,pdf_quality_not_assessed_here,NaN,...,biochemistry;آزمایشگاه;آزمايشگاه,8,4,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
4,5,0024150010_14041209_O_00404121722.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,44.13,extracted_good,,pdf_quality_not_assessed_here,NaN,...,hematology;biochemistry;آزمایشگاه;آزمايشگاه,8,22,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
5,6,0025692283_14041209_O_00404121730.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,42.40,extracted_good,,pdf_quality_not_assessed_here,NaN,...,biochemistry;آزمایشگاه;آزمايشگاه,8,4,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
6,7,0020139871_14041209_O_00404121714.pdf,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.pdf,application/pdf,41.35,extracted_good,,pdf_quality_not_assessed_here,NaN,...,biochemistry;آزمایشگاه;آزمايشگاه,8,4,0,True,True,True,True,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
7,8,20260427_181919.jpg,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.jpg,image/jpeg,58.18,extracted_good,,good_quality,0.765,...,biochemistry;glucose;آزمایشگاه;آزمايشگاه,2,7,0,False,False,False,False,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
8,9,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.jpg,image/jpeg,56.83,needs_manual_review,few_common_fields,good_quality,0.770,...,lab;glucose,1,10,0,False,False,False,False,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
9,10,۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,.jpg,image/jpeg,148.64,extracted_good,,good_quality,0.786,...,laboratory;lab;biochemistry;cbc;wbc;rbc;glucose,2,26,0,False,False,False,False,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...


,part_type,section,field_name,value,result_numeric,unit,reference_range,flag,method,confidence,source_text,index,filename,document_status,document_type,text_path,json_path,test_name_raw
0,common_field,header,center_name,آزمایشگاه پاتوبیولوژی تاو,NaN,NaN,NaN,NaN,NaN,0.70,# آزمایشگاه پاتوبیولوژی تاو#,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
1,common_field,header,national_id,6f7b8e1c3f733b32eaf2a037ee257d451f9423ceb96b2e...,NaN,NaN,NaN,NaN,NaN,0.90,کد ملی :**********,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
2,common_field,header,tracking_number,O-40412-1721,NaN,NaN,NaN,NaN,NaN,0.85,O-40412-1721,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
3,common_field,header,date_of_test_or_report,1404/12/09,NaN,NaN,NaN,NaN,NaN,0.90,تاریخ پذیرش1404/12/09,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
4,common_field,header,patient_name,کورش امینی طھران,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
5,common_field,header,sex,male,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
6,common_field,header,age,34,NaN,NaN,NaN,NaN,NaN,0.85,امینی طھران- آقای کورش- دکتر34 : سن,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
7,common_field,header,doctor_name,امینی طھران- آقای کورش- دکتر34 : سن,NaN,NaN,NaN,NaN,NaN,0.75,پزشک\nامینی طھران- آقای کورش- دکتر34 : سن,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,NaN
8,lab_result,hematology,WBC,7.24,7.24,10^3/uL,3.5-10.5,H,NaN,0.65,W.B.C\n7.24\n10^3/uL\n3.5-10.5\nR.B.C\n4.9\n10...,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,W.B.C
9,lab_result,hematology,RBC,4.9,4.90,10^6/uL,4.32-5.72,H,NaN,0.65,R.B.C\n4.9\n10^6/uL\n4.32 - 5.72\nHemoglobin\n...,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,R.B.C


,index,filename,status,document_type,json_path,text_path
0,1,0014161672_14041209_O_00404121721.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
1,2,0021858845_14041209_O_00404121731.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
2,3,0023471026_14041209_O_00404121728.pdf,needs_manual_review,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
3,4,0025631314_14041209_O_00404121726.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
4,5,0024150010_14041209_O_00404121722.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
5,6,0025692283_14041209_O_00404121730.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
6,7,0020139871_14041209_O_00404121714.pdf,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
7,8,20260427_181919.jpg,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
8,9,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,needs_manual_review,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
9,10,۲۰۲۶۰۴۲۵_۲۲۰۷۰۰.jpg,extracted_good,lab,/Users/aliebrahimi/Documents/Saman Salamat/Ext...,/Users/aliebrahimi/Documents/Saman Salamat/Ext...



## 10.1) Inspect the JSON-like CSV outputs

Use this cell to quickly inspect the nested JSON columns for any file.


In [12]:

# =========================
# 10.1) Inspect JSON-like CSV output
# =========================

if "json_rows_df" in globals() and not json_rows_df.empty:
    CHECK_INDEX = 1  # change this
    row = json_rows_df[json_rows_df["index"] == CHECK_INDEX].iloc[0]
    print("Filename:", row["filename"])
    print("Status:", row["status"])
    print("Document type:", row["document_type"])

    print("\nCOMMON FIELDS JSON:")
    print(json.dumps(json.loads(row["common_fields_json"]), ensure_ascii=False, indent=2)[:3000])

    print("\nLAB RESULTS JSON:")
    print(json.dumps(json.loads(row["lab_results_json"]), ensure_ascii=False, indent=2)[:3000])

    print("\nTEXT PREVIEW:")
    print(row["extracted_text"][:2500])
else:
    print("Run the batch cell first.")


Filename: 0014161672_14041209_O_00404121721.pdf
Status: extracted_good
Document type: lab

COMMON FIELDS JSON:
{
  "center_name": {
    "value": "آزمایشگاه پاتوبیولوژی تاو",
    "confidence": 0.7,
    "source_text": "# آزمایشگاه پاتوبیولوژی تاو#"
  },
  "national_id": {
    "value": "6f7b8e1c3f733b32eaf2a037ee257d451f9423ceb96b2e95426f83929403d282",
    "confidence": 0.9,
    "source_text": "کد ملی :**********"
  },
  "tracking_number": {
    "value": "O-40412-1721",
    "confidence": 0.85,
    "source_text": "O-40412-1721"
  },
  "date_of_test_or_report": {
    "value": "1404/12/09",
    "confidence": 0.9,
    "source_text": "تاریخ پذیرش1404/12/09"
  },
  "patient_name": {
    "value": "کورش امینی طھران",
    "confidence": 0.85,
    "source_text": "امینی طھران- آقای کورش- دکتر34 : سن"
  },
  "sex": {
    "value": "male",
    "confidence": 0.85,
    "source_text": "امینی طھران- آقای کورش- دکتر34 : سن"
  },
  "age": {
    "value": 34,
    "confidence": 0.85,
    "source_text": "امینی طھ


## 11) Review weak files

This cell shows files that were rejected or need manual review.


In [13]:

# =========================
# 11) Weak files review
# =========================

weak_statuses = ["ocr_failed", "poor_quality_rejected", "partial_extraction", "needs_manual_review", "unrelated_or_not_medical", "notebook_error"]
weak_df = summary_df[summary_df["status"].isin(weak_statuses)].copy()
display(weak_df[[
    "index", "filename", "status", "quality_status", "quality_issues",
    "ocr_success", "ocr_text_length", "best_ocr_variant", "best_psm",
    "document_type", "lab_result_count", "warnings", "text_path"
]].head(100))

# Print text preview for worst few
for _, row in weak_df.head(5).iterrows():
    print("="*100)
    print(row["index"], row["filename"], row["status"], row.get("warnings"))
    tp = row.get("text_path")
    if tp and Path(tp).exists():
        print(Path(tp).read_text(encoding="utf-8")[:1500])


,index,filename,status,quality_status,quality_issues,ocr_success,ocr_text_length,best_ocr_variant,best_psm,document_type,lab_result_count,warnings,text_path
2,3,0023471026_14041209_O_00404121728.pdf,needs_manual_review,pdf_quality_not_assessed_here,,True,406,pdf_text_layer,NaN,lab,2,low_structured_data,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
8,9,۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg,needs_manual_review,good_quality,,True,1013,light_sharpen,11.0,lab,10,few_common_fields,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
10,11,20260427_181713.jpg,needs_manual_review,good_quality,,True,235,original_oriented_rot90,6.0,lab,1,low_structured_data,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
11,12,-2147483648_-210195.jpg,needs_manual_review,good_quality,too_bright;possible_rotation_or_sideways,True,425,threshold_fallback,6.0,lab,1,low_structured_data,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
12,13,-2147483648_-210193.jpg,needs_manual_review,good_quality,too_bright;possible_rotation_or_sideways,True,1136,original_oriented,11.0,lab,15,few_common_fields,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
13,14,۲۰۲۶۰۴۲۸_۱۵۰۲۲۸.jpg,needs_manual_review,good_quality,,True,587,original_oriented_rot90,11.0,lab,5,few_common_fields,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
17,18,۲۰۲۶۰۴۲۸_۱۵۰۰۵۱.jpg,needs_manual_review,needs_preprocessing,severe_blur,True,1147,threshold_fallback,6.0,unknown_medical,1,low_structured_data,/Users/aliebrahimi/Documents/Saman Salamat/Ext...
18,19,20260427_181654.jpg,needs_manual_review,needs_preprocessing,,True,290,mild_contrast_rot90,6.0,lab,2,low_structured_data,/Users/aliebrahimi/Documents/Saman Salamat/Ext...


3 0023471026_14041209_O_00404121728.pdf needs_manual_review low_structured_data
# آزمایشگاه پاتوبیولوژی تاو#
 تھران، شھرک غرب، بلوار دریا،نبش چھارراه قدس، پلاک20
021-91012828
O-40412-1728
: پزشک
 انباز- خانم نیلوفر- دکتر25 : سن
شماره :
نام :
کد ملی :0023471026
 - سالFemale
: تاریخ پذیرش1404/12/09 - 14:12:48
Test
Result
Biochemistry
Unit
Normal Range
Flag
Method
Fasting blood sugar
110
mg/dL
70-115
Photometr
This answer is signed out by Dr.N.Adanvar
Print On 1405/01/30-13:48:59 By 2
9 ۲۰۲۶۰۴۲۸_۱۵۰۱۰۶.jpg needs_manual_review few_common_fields
وحدت درمالگاه گام
زماپش
ree.
ITSP pap
تین
Cot myst
5 سوم خیاپان
ین شماره
igh
وخ yy 
13
03137
:
"
دکذر آفای tse ماج
برش ی تار
=
wy
Ue
0
7
hoe
=i
سس
—
Alaa
om
Soil
0
00
jae
111/3
hh
esi
6
Reactive protein (CR)
‘Method
ale
Reference Range
seat
24
1
1
ws
| dist
| TH iodothyra
‘ewe
فلا
سا
eT)
اسر دای
7 WAI; HA
1
18
nwt
Go
Yours! 48194
iN
avian
30
1"
19 Yours (08
11
|
80 Yours (6
‘|
‘Thyroxine Stimulating
80 Yours 4
3
1 Day = 29 Daye 0:76
one
un
HHA HLA
o